# Border-03-TinyML-Training
## Edge AI-Based Acoustic Footstep Detection for Border Surveillance
### Notebook 03 · Model Training, Quantisation & Deployment Profiling

**Pipeline position:** EDA → Preprocessing → **Training ← (you are here)** → Deployment

**Inputs:** `features_train.npz`, `features_val.npz`, `features_test.npz`  
**Outputs:** `border_model.tflite` (INT8), `border_model_float.tflite`, evaluation figures, classification report

**TinyML target:** ESP32 (240 MHz, ~320 KB SRAM, ~4 MB Flash)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings, os, time, itertools, json, struct
warnings.filterwarnings("ignore")

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from tensorflow.keras.utils import to_categorical
print(f"TensorFlow {tf.__version__}  |  Keras {keras.__version__}")

# Sklearn metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_PATH   = "/kaggle/input/borderintrusiondetection-data"
WORKING_PATH = "/kaggle/working"
FIGURES_PATH = "/kaggle/working/figures"
os.makedirs(FIGURES_PATH, exist_ok=True)

# ── Paper-ready plot styling ───────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "#F8F9FA",
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#FFFFFF",
    "grid.linewidth": 1.2,
    "grid.alpha": 0.7,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#CCCCCC",
    "lines.linewidth": 1.8,
    "patch.linewidth": 0.5,
    "savefig.bbox": "tight",
    "savefig.dpi": 300,
    "savefig.facecolor": "white",
})

CLASS_COLORS = {"balastic": "#2196F3", "footsteps": "#FF9800", "noise": "#4CAF50"}
CLASS_NAMES  = ["balastic", "footsteps", "noise"]
PALETTE      = [CLASS_COLORS[c] for c in CLASS_NAMES]

def paper_axes(ax):
    ax.set_facecolor("#F8F9FA")
    for spine in ax.spines.values():
        spine.set_edgecolor("#CCCCCC")
        spine.set_linewidth(0.8)
    ax.grid(True, color="white", linewidth=1.2, alpha=0.7)
    ax.tick_params(colors="#444444")
    ax.title.set_color("#222222")
    ax.xaxis.label.set_color("#444444")
    ax.yaxis.label.set_color("#444444")
    return ax

print("✓ Styling loaded  |  TF", tf.__version__)